# Project 2

##  Route Optimization Engine — Supply Chain Analytics

In [2]:
pip install faker pandas numpy

Note: you may need to restart the kernel to use updated packages.


In [3]:
from faker import Faker
import pandas as pd
import numpy as np
import random

In [4]:
warehouse = {
    "customer_id": 0,
    "name": "Central Warehouse",
    "latitude": 28.6139,
    "longitude": 77.2090,
    "demand": 0   
}

In [5]:
customers=[]
faker=Faker()

In [6]:
warehouse_lat=28.6139
warehouse_long=77.2090

In [7]:
for i in range(1,51):
    customer = {
        "customer_id": i,
        "name": faker.name(),
        "latitude": round(warehouse_lat+random.uniform(-0.2,0.2),6),
        "longitude": round(warehouse_long+random.uniform(-0.2,0.2),6),
        "demand": random.randint(1,10)
    }
    customers.append(customer)

In [8]:
df=pd.DataFrame(customers)

In [9]:
df.head()

,customer_id,name,latitude,longitude,demand
0,1,Lorraine Jones,28.771541,77.136946,4
1,2,Paul Wright,28.426535,77.012947,5
2,3,Michael Smith,28.623484,77.055727,3
3,4,Rachel Cooke,28.459673,77.017504,2
4,5,John Kelley,28.440967,77.301528,7


In [10]:
df.to_csv("customers.csv",index=False)

In [11]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 5 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   customer_id  50 non-null     int64  
 1   name         50 non-null     str    
 2   latitude     50 non-null     float64
 3   longitude    50 non-null     float64
 4   demand       50 non-null     int64  
dtypes: float64(2), int64(2), str(1)
memory usage: 2.7 KB


In [21]:
df.describe()

,customer_id,latitude,longitude,demand
count,50.00000,50.000000,50.000000,50.000000
mean,25.50000,28.617395,77.206657,5.160000
std,14.57738,0.110943,0.118589,3.106018
min,1.00000,28.425209,77.012065,1.000000
25%,13.25000,28.542202,77.104090,3.000000
50%,25.50000,28.619372,77.206834,4.000000
75%,37.75000,28.697419,77.303461,8.000000
max,50.00000,28.806128,77.403911,10.000000


In [13]:
df.isnull().sum()

customer_id    0
name           0
latitude       0
longitude      0
demand         0
dtype: int64

In [14]:
import sqlite3

In [25]:
conn=sqlite3.connect("route_optimization.db")

In [26]:
df.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

50

In [27]:
query = """
SELECT 
customer_id, name, latitude, longitude, demand,
NTILE(5) OVER(ORDER BY latitude) zone
FROM customers
"""

zoned_df= pd.read_sql(query, conn)

zoned_df.head()

,customer_id,name,latitude,longitude,demand,zone
0,50,Carolyn Klein,28.415931,77.353636,5,1
1,25,Andrea Cook,28.424612,77.244359,10,1
2,2,Paul Wright,28.426535,77.012947,5,1
3,13,Cheyenne Walker,28.435633,77.367843,10,1
4,5,John Kelley,28.440967,77.301528,7,1


In [28]:
locations = zoned_df[["latitude", "longitude"]].to_numpy()
locations.shape

(50, 2)

In [29]:
warehouse_location = np.array([
    warehouse_lat,
    warehouse_long
])

locations = np.vstack([
    warehouse_location,
    locations
])
locations.shape

(51, 2)

In [30]:
def haversine_distance(lat1, long1, lat2, long2):
    R = 6371 

    lat1 = np.radians(lat1)
    long1 = np.radians(long1)
    lat2 = np.radians(lat2)
    long2 = np.radians(long2)

    dlat = lat2 - lat1
    dlong = long2 - long1

    a = (
        np.sin(dlat/2) ** 2
        + np.cos(lat1)
        * np.cos(lat2)
        * np.sin(dlong/2) ** 2
    )

    c = 2 * np.arcsin(np.sqrt(a))

    return R * c

In [31]:
n =len(locations)

distance_matrix = np.zeros((n, n))

for i in range(n):
    for j in range(n):

        distance_matrix[i][j] = haversine_distance(
            locations[i][0],
            locations[i][1],
            locations[j][0],
            locations[j][1]            
        )

distance_matrix.shape

(51, 51)

In [32]:
distance_matrix[:5, :5]

array([[ 0.        , 26.15887976, 21.32949191, 28.30116965, 25.17439666],
       [26.15887976,  0.        , 10.73011296, 33.33779624,  2.59413729],
       [21.32949191, 10.73011296,  0.        , 22.63051849, 12.1368658 ],
       [28.30116965, 33.33779624, 22.63051849,  0.        , 34.71779233],
       [25.17439666,  2.59413729, 12.1368658 , 34.71779233,  0.        ]])

In [34]:
baseline_distance = 0

for i in range(50):
    baseline_distance += distance_matrix[i][i+1]

baseline_distance += distance_matrix[50][0]

In [35]:
demands = [0] + zoned_df["demand"].tolist()

vehicle_capacity = 40

import math

num_vehicles = math.ceil(
    sum(demands) / vehicle_capacity
)

unvisited = set(range(1, len(locations)))
routes = []

print("Total demand:", sum(demands))
print("Vehicle capacity:", vehicle_capacity)
print("Number of vehicles:", num_vehicles)
print("Customers to visit:", len(unvisited))

Total demand: 277
Vehicle capacity: 40
Number of vehicles: 7
Customers to visit: 50


In [36]:
routes = []

for vehicle in range(num_vehicles):

    current = 0
    current_load = 0
    route = [0]

    while unvisited:

        feasible_customers = [
            customer
            for customer in unvisited
            if current_load + demands[customer] <= vehicle_capacity
        ]

        if not feasible_customers:
            break

        next_customer = min(
            feasible_customers,
            key=lambda customer: distance_matrix[current][customer]
        )

        route.append(next_customer)

        current_load += demands[next_customer]

        unvisited.remove(next_customer)

        current = next_customer

    route.append(0)

    routes.append({
        "vehicle": vehicle + 1,
        "route": route,
        "load": current_load
    })

In [37]:
for route in routes:
    print(
        f"Vehicle {route['vehicle']}: "
        f"{route['route']} | "
        f"Load = {route['load']}"
    )

print("Remaining customers:", unvisited)

Vehicle 1: [0, 31, 25, 23, 20, 15, 13, 14, 9, 0] | Load = 40
Vehicle 2: [0, 29, 26, 24, 17, 18, 16, 11, 0] | Load = 40
Vehicle 3: [0, 35, 39, 44, 46, 45, 43, 0] | Load = 40
Vehicle 4: [0, 28, 32, 33, 27, 22, 21, 10, 37, 0] | Load = 40
Vehicle 5: [0, 30, 34, 36, 48, 50, 47, 49, 0] | Load = 40
Vehicle 6: [0, 12, 7, 2, 6, 5, 1, 0] | Load = 39
Vehicle 7: [0, 40, 42, 41, 38, 19, 4, 8, 3, 0] | Load = 38
Remaining customers: set()


In [38]:
def calculate_route_distance(route):
    total_distance = 0

    for i in range(len(route) - 1):
        total_distance += distance_matrix[
            route[i]
        ][
            route[i + 1]
        ]

    return total_distance

for route in routes:

    route["distance_km"] = calculate_route_distance(
        route["route"]
    )

    print(
        f"Vehicle {route['vehicle']}: "
        f"Distance = {route['distance_km']:.2f} km | "
        f"Load = {route['load']}"
    )

Vehicle 1: Distance = 45.56 km | Load = 40
Vehicle 2: Distance = 44.96 km | Load = 40
Vehicle 3: Distance = 39.98 km | Load = 40
Vehicle 4: Distance = 95.42 km | Load = 40
Vehicle 5: Distance = 84.05 km | Load = 40
Vehicle 6: Distance = 69.92 km | Load = 39
Vehicle 7: Distance = 120.81 km | Load = 38


In [39]:
optimized_distance = sum(
    route["distance_km"]
    for route in routes
)

print(f"Optimized distance: {optimized_distance:.2f} km")

Optimized distance: 500.71 km


In [59]:
print(f"Baseline distance: {baseline_distance:.2f} km")
print(f"Optimized distance: {optimized_distance:.2f} km")

Baseline distance: 629.11 km
Optimized distance: 484.61 km


In [60]:
distance_saved = baseline_distance - optimized_distance

reduction_percentage = (
    distance_saved / baseline_distance
) * 100

print(f"Distance saved: {distance_saved:.2f} km")
print(f"Distance reduction: {reduction_percentage:.2f}%")

Distance saved: 144.50 km
Distance reduction: 22.97%


In [42]:
def two_opt(route, distance_matrix):

    best_route = route.copy()
    best_distance = calculate_route_distance(best_route)

    improved = True

    while improved:

        improved = False

        for i in range(1, len(best_route) - 2):

            for j in range(i + 1, len(best_route) - 1):

                new_route = (
                    best_route[:i]
                    + best_route[i:j + 1][::-1]
                    + best_route[j + 1:]
                )

                new_distance = calculate_route_distance(new_route)

                if new_distance < best_distance:

                    best_route = new_route
                    best_distance = new_distance

                    improved = True

    return best_route, best_distance

In [43]:
for route in routes:

    improved_route, improved_distance = two_opt(
        route["route"],
        distance_matrix
    )

    route["route"] = improved_route
    route["distance_km"] = improved_distance

In [44]:
for route in routes:

    print(
        f"Vehicle {route['vehicle']}: "
        f"{route['route']} | "
        f"Distance = {route['distance_km']:.2f} km | "
        f"Load = {route['load']}"
    )

Vehicle 1: [0, 31, 23, 20, 15, 13, 9, 14, 25, 0] | Distance = 39.63 km | Load = 40
Vehicle 2: [0, 16, 11, 18, 17, 24, 26, 29, 0] | Distance = 41.80 km | Load = 40
Vehicle 3: [0, 43, 45, 46, 44, 39, 35, 0] | Distance = 39.98 km | Load = 40
Vehicle 4: [0, 37, 33, 32, 28, 27, 22, 21, 10, 0] | Distance = 90.98 km | Load = 40
Vehicle 5: [0, 30, 34, 36, 48, 50, 47, 49, 0] | Distance = 84.05 km | Load = 40
Vehicle 6: [0, 12, 7, 2, 6, 5, 1, 0] | Distance = 69.92 km | Load = 39
Vehicle 7: [0, 42, 40, 41, 38, 19, 4, 3, 8, 0] | Distance = 118.25 km | Load = 38


In [45]:
optimized_distance = sum(
    route["distance_km"]
    for route in routes
)

print(f"Final optimized distance: {optimized_distance:.2f} km")

Final optimized distance: 484.61 km


In [50]:
distance_saved = baseline_distance - optimized_distance

reduction_percentage = (
    distance_saved / baseline_distance
) * 100

print(f"Baseline distance: {baseline_distance:.2f} km")
print(f"Optimized distance: {optimized_distance:.2f} km")
print(f"Distance saved: {distance_saved:.2f} km")
print(f"Reduction: {reduction_percentage:.2f}%")

Baseline distance: 629.11 km
Optimized distance: 484.61 km
Distance saved: 144.50 km
Reduction: 22.97%


In [53]:
!pip install folium

import folium

route_map = folium.Map(
    location=[warehouse_lat, warehouse_long],
    zoom_start=12
)

folium.Marker(
    [warehouse_lat, warehouse_long],
    popup="Central Warehouse"
).add_to(route_map)

for i in range(1, len(locations)):

    folium.Marker(
        [locations[i][0], locations[i][1]],
        popup=f"Customer {i} | Demand: {demands[i]}"
    ).add_to(route_map)

for route in routes:

    route_coordinates = [
        [locations[node][0], locations[node][1]]
        for node in route["route"]
    ]

    folium.PolyLine(
        route_coordinates,
        weight=4,
        popup=f"Vehicle {route['vehicle']}"
    ).add_to(route_map)

route_map.save("optimized_routes.html")

route_map